# Experiment 3: Curriculum vs Hardening

Does hardening enable progressive stacking without data curriculum?

**Prerequisites**: Runtime > Change runtime type > **GPU (T4)**

| Condition | Hardening | Data order | Tests |
|-----------|-----------|------------|-------|
| SoftGear + random | gradual | shuffle | Hardening effect |
| No protection + random | none | shuffle | Baseline |
| No protection + curriculum | none | sorted (easy first) | Curriculum effect |
| SoftGear + curriculum | gradual | sorted (easy first) | Combined effect |

## Setup

In [ ]:
!pip install -q git+https://github.com/byExist/softgear.git

In [ ]:
!softgear download sudoku9

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU not available! Change runtime type."
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# type: ignore
from google.colab import drive

drive.mount("/content/drive")
CKPT_ROOT = "/content/drive/MyDrive/softgear/checkpoints"

## Training

In [ ]:
COMMON = "--task sudoku9 --num-gears 7 --hidden-dim 128 --ffn-dim 512 --num-heads 4 --patience 5 --max-total-steps 40000 --max-samples 50000 --seed 42"

In [ ]:
# SoftGear + random (= exp1-gradual)
!softgear train --hardening gradual --no-curriculum --checkpoint-dir {CKPT_ROOT}/exp3-gradual-random $COMMON

In [ ]:
# No protection + random (= exp1-none)
!softgear train --hardening none --no-curriculum --checkpoint-dir {CKPT_ROOT}/exp3-none-random $COMMON

In [ ]:
# No protection + curriculum
!softgear train --hardening none --curriculum --checkpoint-dir {CKPT_ROOT}/exp3-none-curriculum $COMMON

In [ ]:
# SoftGear + curriculum
!softgear train --hardening gradual --curriculum --checkpoint-dir {CKPT_ROOT}/exp3-gradual-curriculum $COMMON

## Results

In [ ]:
import torch
import pandas as pd
from typing import Any
from softgear.tasks.registry import get_task
from softgear.config import DataConfig, ModelConfig

task = get_task("sudoku9")
data_cfg = DataConfig(path="data/sudoku-extreme", batch_size=64, num_workers=2, max_samples=None, curriculum=False)
_, val_loader = task.build_loaders(data_cfg)
device = torch.device("cuda")

experiments = {
    "gradual+random": f"{CKPT_ROOT}/exp3-gradual-random",
    "none+random": f"{CKPT_ROOT}/exp3-none-random",
    "none+curriculum": f"{CKPT_ROOT}/exp3-none-curriculum",
    "gradual+curriculum": f"{CKPT_ROOT}/exp3-gradual-curriculum",
}

results: dict[str, Any] = {}
for name, ckpt_dir in experiments.items():
    ckpt = torch.load(f"{ckpt_dir}/best.pt", map_location=device, weights_only=False)
    cfg = ckpt["config"]

    model_cfg = ModelConfig(**cfg["model"])
    model = task.build_model(model_cfg)
    task.mount_all_gears(model, model_cfg)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()

    all_preds: list[torch.Tensor] = []
    all_targets: list[torch.Tensor] = []
    all_inputs: list[torch.Tensor] = []
    with torch.no_grad():
        for batch in val_loader:
            x, y = batch[0].to(device), batch[1].to(device)
            preds = task.predict_fn(model(x).logits)
            all_preds.append(preds)
            all_targets.append(y)
            all_inputs.append(x)

    metrics = task.metrics_fn(torch.cat(all_preds), torch.cat(all_targets), torch.cat(all_inputs))
    results[name] = {
        "phase": ckpt["phase"],
        "global_step": ckpt.get("global_step", "?"),
        "val_loss": ckpt["best_val_loss"],
        **metrics,
    }

df = pd.DataFrame(results).T
df.index.name = "condition"
df